# Phase 9 — Question Classifier

This notebook implements the question-classification layer
of the Databricks GenAI Data Analyst Copilot.

The classifier determines whether a user question requires:

1. SQL analytics
2. RAG/document retrieval
3. Hybrid SQL + RAG processing
4. Unsupported-question handling

Architecture:

User Question
      |
      v
Question Classifier
      |
      +----> SQL
      |
      +----> RAG
      |
      +----> Hybrid
      |
      +----> Unsupported

Important:

The classifier does not generate SQL.

It only determines the appropriate downstream processing path.

In [0]:
import re

print("Question Classifier initialized.")

In [0]:
# ============================================================
# SQL ANALYTICAL INTENTS
# ============================================================

SQL_INTENT_KEYWORDS = {

    "aggregation": [
        "total",
        "sum",
        "average",
        "avg",
        "mean",
        "count",
        "how much",
        "how many",
    ],

    "comparison": [
        "compare",
        "comparison",
        "versus",
        "vs",
        "difference between",
        "higher than",
        "lower than",
    ],

    "trend": [
        "trend",
        "over time",
        "monthly",
        "weekly",
        "daily",
        "month over month",
        "mom",
        "growth",
        "change over time",
    ],

    "ranking": [
        "top",
        "bottom",
        "highest",
        "lowest",
        "best",
        "worst",
        "most",
        "least",
    ],

    "filtering": [
        "where",
        "in japan",
        "in usa",
        "in france",
        "in germany",
        "for customer",
        "for region",
        "for category",
        "orders from",
        "sales from",
    ],

    "anomaly": [
        "anomaly",
        "anomalies",
        "unusual",
        "unexpected",
        "decline",
        "decrease",
        "drop",
        "spike",
        "why did",
        "why has",
        "why was",
    ],

    "descriptive_statistics": [
        "median",
        "standard deviation",
        "std",
        "variance",
        "distribution",
        "percentile",
        "statistics",
        "statistical",
    ],

    "visualization": [
        "show me a chart",
        "show chart",
        "visualize",
        "visualization",
        "plot",
        "graph",
        "chart",
    ],
}

print("SQL intent categories:")
for intent in SQL_INTENT_KEYWORDS:
    print("-", intent)

In [0]:
# ============================================================
# RAG / DOCUMENT PATTERNS
# ============================================================

RAG_PATTERNS = [

    r"\bwhat is the .*policy\b",
    r"\bwhat is the .*policies\b",

    r"\bwhat does the policy say\b",
    r"\bwhat does the document say\b",

    r"\bwhat are the .*guidelines\b",
    r"\bwhat are the rules\b",

    r"\bwhat is the discount policy\b",
    r"\bwhat is the return policy\b",
    r"\bwhat is the refund policy\b",
    r"\bwhat is the cancellation policy\b",
    r"\bwhat is the sales policy\b",
    r"\bwhat is the product policy\b",

    r"\bwhat are the terms\b",

    r"\baccording to the policy\b",
    r"\baccording to the document\b",
    r"\baccording to the guidelines\b",

    r"\bexplain the policy\b",
    r"\bexplain the guidelines\b",
    r"\bexplain the document\b",

]

print("RAG patterns:", len(RAG_PATTERNS))

In [0]:
# ============================================================
# HYBRID SQL + RAG PATTERNS
# ============================================================

HYBRID_PATTERNS = [

    r"\brevenue\b.*\bpolicy\b",
    r"\bsales\b.*\bpolicy\b",
    r"\bprofit\b.*\bpolicy\b",

    r"\bdiscount\b.*\bsales\b",
    r"\bsales\b.*\bdiscount\b",

    r"\brevenue\b.*\bguideline\b",
    r"\bsales\b.*\bguideline\b",

    r"\bcompare\b.*\bpolicy\b",

    r"\bhow did\b.*\bpolicy\b",

    r"\bwas .* related to\b",

    r"\brelated to the .*policy\b",

]

print("Hybrid patterns:", len(HYBRID_PATTERNS))

In [0]:
# ============================================================
# UNSUPPORTED QUESTION PATTERNS
# ============================================================

UNSUPPORTED_PATTERNS = [

    r"\bweather\b",
    r"\bmovie\b",
    r"\bsports\b",
    r"\bfootball\b",
    r"\bcricket\b",
    r"\bstock price\b",
    r"\bstock market\b",
    r"\bpolitics\b",

]

print("Unsupported patterns:", len(UNSUPPORTED_PATTERNS))

In [0]:
# ============================================================
# HELPER FUNCTION
# ============================================================

def matches_pattern(question, patterns):
    """
    Returns True when the question matches
    at least one regex pattern.
    """

    return any(
        re.search(pattern, question)
        for pattern in patterns
    )


print("Pattern matching helper ready.")

In [0]:
# ============================================================
# QUESTION NORMALIZATION
# ============================================================

def normalize_question(question):
    """
    Normalize user input for consistent classification.
    """

    if not isinstance(question, str):
        return ""

    question = question.lower().strip()

    # Replace multiple whitespace characters
    question = re.sub(r"\s+", " ", question)

    return question


# Test normalization

test_question = "   What   is   the discount policy?   "

print("Original:")
print(test_question)

print("\nNormalized:")
print(normalize_question(test_question))

In [0]:
# ============================================================
# MAIN CLASSIFIER
# ============================================================

def classify_question(question):

    normalized_question = normalize_question(question)

    # --------------------------------------------------------
    # Invalid question
    # --------------------------------------------------------

    if not normalized_question:

        return {
            "question": question,
            "normalized_question": "",
            "intent": "unsupported",
            "requires_sql": False,
            "requires_rag": False,
            "requires_hybrid": False,
            "confidence": 1.0,
            "scores": {},
            "reason": "Question is empty or invalid."
        }

    # --------------------------------------------------------
    # STEP 1 — HYBRID
    #
    # Hybrid must be checked BEFORE RAG and SQL because
    # hybrid questions contain both analytical and document
    # requirements.
    # --------------------------------------------------------

    if matches_pattern(
        normalized_question,
        HYBRID_PATTERNS
    ):

        return {
            "question": question,
            "normalized_question": normalized_question,
            "intent": "hybrid",
            "requires_sql": True,
            "requires_rag": True,
            "requires_hybrid": True,
            "confidence": 0.95,
            "scores": {
                "hybrid": 1
            },
            "reason": (
                "Question requires both analytical data "
                "and business-document context."
            )
        }

    # --------------------------------------------------------
    # STEP 2 — RAG
    #
    # This must happen before SQL keyword scoring.
    #
    # Example:
    #
    # What is the discount policy?
    #
    # The word "discount" must NOT cause SQL routing.
    # --------------------------------------------------------

    if matches_pattern(
        normalized_question,
        RAG_PATTERNS
    ):

        return {
            "question": question,
            "normalized_question": normalized_question,
            "intent": "rag",
            "requires_sql": False,
            "requires_rag": True,
            "requires_hybrid": False,
            "confidence": 0.97,
            "scores": {
                "rag": 1
            },
            "reason": (
                "Question asks about a business policy, "
                "guideline, document, or business rule."
            )
        }

    # --------------------------------------------------------
    # STEP 3 — UNSUPPORTED
    # --------------------------------------------------------

    if matches_pattern(
        normalized_question,
        UNSUPPORTED_PATTERNS
    ):

        return {
            "question": question,
            "normalized_question": normalized_question,
            "intent": "unsupported",
            "requires_sql": False,
            "requires_rag": False,
            "requires_hybrid": False,
            "confidence": 0.95,
            "scores": {
                "unsupported": 1
            },
            "reason": (
                "Question is outside the supported "
                "business analytics and document scope."
            )
        }

    # --------------------------------------------------------
    # STEP 4 — SQL INTENT SCORING
    # --------------------------------------------------------

    scores = {}

    for intent, keywords in SQL_INTENT_KEYWORDS.items():

        score = 0

        for keyword in keywords:

            if keyword in normalized_question:
                score += 1

        scores[intent] = score

    # Remove zero scores

    non_zero_scores = {
        intent: score
        for intent, score in scores.items()
        if score > 0
    }

    # --------------------------------------------------------
    # STEP 5 — NO MATCH
    # --------------------------------------------------------

    if not non_zero_scores:

        return {
            "question": question,
            "normalized_question": normalized_question,
            "intent": "unsupported",
            "requires_sql": False,
            "requires_rag": False,
            "requires_hybrid": False,
            "confidence": 0.60,
            "scores": scores,
            "reason": (
                "Question does not match a supported "
                "analytical or document-question pattern."
            )
        }

    # --------------------------------------------------------
    # STEP 6 — BEST SQL INTENT
    # --------------------------------------------------------

    best_intent = max(
        non_zero_scores,
        key=non_zero_scores.get
    )

    best_score = non_zero_scores[best_intent]

    confidence = min(
        0.95,
        0.70 + (best_score * 0.10)
    )

    return {
        "question": question,
        "normalized_question": normalized_question,
        "intent": best_intent,
        "requires_sql": True,
        "requires_rag": False,
        "requires_hybrid": False,
        "confidence": confidence,
        "scores": scores,
        "reason": (
            f"Question classified as '{best_intent}' "
            "based on analytical language."
        )
    }


print("Main classifier ready.")

In [0]:
result = classify_question(
    "What is the discount policy?"
)

print(result)

In [0]:
{
    "question": "What is the discount policy?",
    "normalized_question": "what is the discount policy?",
    "intent": "rag",
    "requires_sql": False,
    "requires_rag": True,
    "requires_hybrid": False,
    "confidence": 0.97
}


In [0]:
test_questions = [

    # SQL
    "What is total revenue?",
    "Which region generated the highest revenue?",
    "What are the top 10 customers by revenue?",
    "What is monthly revenue?",
    "Which category has the highest profit margin?",
    "What is the average order value?",
    "Which region has the highest number of orders?",
    "How did revenue change month over month?",
    "Which customers generated the most profit?",
    "Show revenue by sales channel.",

    # RAG
    "What is the discount policy?",
    "What is the return policy?",

    # Hybrid
    "Compare Japanese sales with the company discount policy.",

    # Unsupported
    "What is the weather today?"
]


for question in test_questions:

    result = classify_question(question)

    print("=" * 80)
    print("Question:", question)
    print("Intent:", result["intent"])
    print("SQL:", result["requires_sql"])
    print("RAG:", result["requires_rag"])
    print("Hybrid:", result["requires_hybrid"])
    print("Confidence:", result["confidence"])
    print("Reason:", result["reason"])

In [0]:
test_results = []

for question in test_questions:

    result = classify_question(question)

    test_results.append({
        "question": question,
        "intent": result["intent"],
        "requires_sql": result["requires_sql"],
        "requires_rag": result["requires_rag"],
        "requires_hybrid": result["requires_hybrid"],
        "confidence": result["confidence"]
    })


display(
    spark.createDataFrame(test_results)
)

In [0]:
expected_results = {

    "What is total revenue?": "aggregation",

    "Which region generated the highest revenue?": "ranking",

    "What are the top 10 customers by revenue?": "ranking",

    "What is monthly revenue?": "trend",

    "Which category has the highest profit margin?": "ranking",

    "What is the average order value?": "aggregation",

    "Which region has the highest number of orders?": "ranking",

    "How did revenue change month over month?": "trend",

    "Which customers generated the most profit?": "ranking",

    "Show revenue by sales channel.": "visualization",

    "What is the discount policy?": "rag",

    "What is the return policy?": "rag",

    "Compare Japanese sales with the company discount policy.": "hybrid",

    "What is the weather today?": "unsupported",
}


validation_results = []

for question, expected_intent in expected_results.items():

    result = classify_question(question)

    actual_intent = result["intent"]

    status = (
        "PASS"
        if actual_intent == expected_intent
        else "FAIL"
    )

    validation_results.append({
        "question": question,
        "expected_intent": expected_intent,
        "actual_intent": actual_intent,
        "status": status
    })


validation_df = spark.createDataFrame(
    validation_results
)

display(validation_df)

In [0]:
total_tests = len(validation_results)

passed_tests = sum(
    1
    for result in validation_results
    if result["status"] == "PASS"
)

failed_tests = total_tests - passed_tests

accuracy = (
    passed_tests / total_tests
    if total_tests > 0
    else 0
)

print("=" * 60)
print("QUESTION CLASSIFIER VALIDATION")
print("=" * 60)

print(f"Total tests: {total_tests}")
print(f"Passed: {passed_tests}")
print(f"Failed: {failed_tests}")
print(f"Accuracy: {accuracy:.2%}")

print("=" * 60)

if failed_tests == 0:

    print("✓ ALL CLASSIFICATION TESTS PASSED")

else:

    print("✗ SOME CLASSIFICATION TESTS FAILED")

In [0]:
routing_tests = [

    {
        "question": "What is the discount policy?",
        "expected": "rag"
    },

    {
        "question": "What was the total revenue in Japan?",
        "expected": "aggregation"
    },

    {
        "question": (
            "Revenue in Japan was low. "
            "Was this related to the company's discount policy?"
        ),
        "expected": "hybrid"
    }
]


print("=" * 70)
print("CRITICAL ROUTING TESTS")
print("=" * 70)


for test in routing_tests:

    result = classify_question(
        test["question"]
    )

    actual = result["intent"]

    status = (
        "PASS"
        if actual == test["expected"]
        else "FAIL"
    )

    print("\nQuestion:")
    print(test["question"])

    print("Expected:", test["expected"])
    print("Actual:", actual)
    print("Status:", status)

One important thing about this version

We are deliberately not using an LLM for classification yet.

That's actually a good engineering decision.

The architecture becomes:

User Question
      │
      ▼
Rule-based Question Classifier
      │
      ├───────────────┐
      │               │
      ▼               ▼
     SQL             RAG
      │               │
      └───────┬───────┘
              ▼
           Hybrid

Later, we can improve this with an LLM-based fallback:

                 User Question
                       │
                       ▼
              Rule-based Router
                       │
              ┌────────┴────────┐
              │                 │
         High confidence    Low confidence
              │                 │
              ▼                 ▼
           Route             LLM Classifier
                                │
                                ▼
                              Route

This is more production-oriented than blindly asking an LLM to classify every question.